# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```plaintext
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

The dataset contains ordered logistic regression outputs and survey data about household adoption of indigenous and modern knowledge in rangeland management practices in Northern Kenya.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}: {metadata['description']}")
print(f"Published: {metadata['datePublished']}")
print(f"Identifier: {metadata['identifier']}")
print(f"Authors: {[author['@id'] for author in metadata['author']]}")
print(f"Keywords: {metadata.get('keywords', [])}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities—record sets, fields, columns—are referenced by their `@id`, per the Croissant schema.

In [ ]:
# List available record sets
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record sets.")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}, name: {rs.get('name', '')}")
    if 'fields' in rs:
        print("  Fields:")
        for f in rs['fields']:
            print(f"    Field @id: {f['@id']}, name: {f.get('name', '')}, dataType: {f.get('dataType', '')}")
    if 'columns' in rs:
        print("  Columns:")
        for c in rs['columns']:
            print(f"    Column @id: {c['@id']}, name: {c.get('name', '')}, dataType: {c.get('dataType', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set using their @id
# Grab the first record set @id for demo
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet @id '{record_set_id}'")

# Display columns of the first available DataFrame
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Columns for RecordSet @id {first_rs_id}: {dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All references are by `@id`.

In [ ]:
# Select a numeric field for analysis using @id

# We'll attempt to select a numeric field 'log_likelihood' as an example
numeric_field_id = None
group_field_id = None

# Try to find relevant field IDs
if dataframes:
    df = dataframes[first_rs_id]
    # Try standard names for logistic regression outputs
    for col in df.columns:
        if 'log_likelihood' in col.lower():
            numeric_field_id = col
        if 'ward' in col.lower() or 'county' in col.lower():
            group_field_id = col

    if numeric_field_id is not None:
        threshold = df[numeric_field_id].mean()  # Use mean as threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by ward or county if available
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No group field found in columns.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No dataframes loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id} in RecordSet {first_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group field available, boxplot
    if group_field_id is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field or dataframe found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded metadata and record sets using `mlcroissant`.
- Identified available fields and columns by their `@id`.
- Demonstrated EDA using numeric and grouping fields.
- Visualized field distributions and relationships by group.

Further exploration can include more advanced analytics, visualizations, or export for reproducible research and policy insights on knowledge adoption in rangeland management.